# Importing Libraries

In [ ]:
import hickle
import pandas as pd
import scipy.signal
# import shap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import FormatStrFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# from astropy.table import Table
from scipy import ndimage
from tqdm import tqdm
import corner
from PIL import Image
import io

import os

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

from scipy.stats import chi2
from sklearn.datasets import make_s_curve
import matplotlib as mpl
import matplotlib.pyplot as plt
import tqdm

import ili
from ili.dataloaders import NumpyLoader
from ili.inference import InferenceRunner
from ili.validation.metrics import PosteriorCoverage, PlotSinglePosterior

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

from datasets import load_dataset, DatasetDict
from transformers import ViTFeatureExtractor, ViTForImageClassification, ViTModel, AdamW, AutoImageProcessor, AutoModel
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
from tqdm import tqdm

# import dense_basis as db

Device: cuda
Device: cuda


# Data Importing

In [ ]:
def check_degeneracy(samples, vb=False):
    # Ensure only the 6 physical parameters are used if 34D samples are passed
    samples_6d = samples[:, 0:6] if samples.shape[1] > 6 else samples
    covs = np.zeros((samples_6d.shape[1], samples_6d.shape[1]))
    pvals = np.ones((samples_6d.shape[1], samples_6d.shape[1]))
    for i in range(samples_6d.shape[1]):
        for j in range(i):
            covs[i,j], pvals[i,j] = spearmanr(samples_6d[:,i], samples_6d[:,j])

    if vb==True:
        plt.imshow(np.abs(covs[1:,:-1]));
        plt.xticks(np.arange(5), labels[0:5])
        plt.yticks(np.arange(5), labels[1:])
        plt.colorbar(); plt.show()
        
        plt.imshow(np.abs((pvals<0.05)[1:,:-1]))
        plt.xticks(np.arange(5), labels[0:5])
        plt.yticks(np.arange(5), labels[1:])
        plt.colorbar(); plt.show()

    return covs[1:,:-1], pvals[1:,:-1]

In [ ]:
GSWLC_data = load_dataset('juanpabloalfonzo/GSWLC_Data',split='train')
gswlc_data = GSWLC_data.to_pandas()

gswlc_images_skyserver = load_dataset('juanpabloalfonzo/GSWLC_SkyServer',split='train')
gswlc_images_skyserver = gswlc_images_skyserver['images'].copy()

timescales_data_cami_fixed = np.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/all_timescales_cami_fixed_v2.npy', allow_pickle=True).item()


train_index = int(0.8*len(gswlc_data))
sample_size = len(gswlc_data) - train_index

In [ ]:
morph_data = load_dataset('juanpabloalfonzo/GSWLC_morph',split='train')
morph_data = morph_data.to_pandas()

if 'ID' not in gswlc_data.columns:
        print("\nError: 'ID' column not found in gswlc_data.")
elif 'ID' not in morph_data.columns:
    print("\nError: 'ID' column not found in morph_data.")
# Check if 'sersic_n_morph' exists in morph_data
elif 'bt_morph' not in morph_data.columns:
        print("\nError: 'bt_morph' column not found in morph_data.")
else:
    # Optional: Ensure 'ID' columns have the same data type if needed
    # gswlc_data['ID'] = gswlc_data['ID'].astype(int)
    # morph_data['ID'] = morph_data['ID'].astype(int)
    print("\nData loaded successfully. Proceeding with merge...")

    # --- Perform the left merge ---
    # We select only the 'ID' and 'sersic_n_morph' columns from morph_data
    gswlc_data_merged = pd.merge(gswlc_data, morph_data[['ID', 'pbulge_morph','bt_morph','half_light_radius_morph', 'bulge_n_morph', 'disk_n_morph', 'ttype_morph', 'pbar_morph', 'pdisk_morph', 'pedgeon_morph', 'pmerger_morph']], on='ID', how='left')

    # --- Display results ---
    print("\nMerge complete. Updated gswlc_data DataFrame head:")
    print(gswlc_data_merged.head())

    # Check for NaN values in the new column
    print("\nNumber of NaN values in 'bulge_n_morph' after merge:")
    print(gswlc_data_merged['bulge_n_morph'].isnull().sum())

    # You might want to assign the merged result back or use the new DataFrame:
    # gswlc_data = gswlc_data_merged

sersic_n = np.array(gswlc_data_merged['bulge_n_morph'][train_index:])
bt = np.array(gswlc_data_merged['bt_morph'][train_index:])
pbulge = np.array(gswlc_data_merged['pbulge_morph'][train_index:])
ttype= np.array(gswlc_data_merged['ttype_morph'][train_index:])
pbar = np.array(gswlc_data_merged['pbar_morph'][train_index:])
pdisk = np.array(gswlc_data_merged['pdisk_morph'][train_index:])
pedgeon = np.array(gswlc_data_merged['pedgeon_morph'][train_index:])
pmerger = np.array(gswlc_data_merged['pmerger_morph'][train_index:])


In [ ]:
temp_fuv = np.array(gswlc_data['FUV']).copy()
temp_fuv[temp_fuv > 25] = 25 + 0.3*np.random.normal(size=(np.sum(temp_fuv>25),))

temp_w2 = np.array(gswlc_data['W2']).copy()
temp_w2[temp_w2 > 23] = 23

temp_w3 = np.array(gswlc_data['W3']).copy()
temp_w3[temp_w3 > 23] = 23 + 0.3*np.random.normal(size=(np.sum(temp_w3>23),))

temp_w4 = np.array(gswlc_data['W4']).copy()
temp_w4[temp_w4 > 23] = 23 + 0.3*np.random.normal(size=(np.sum(temp_w4>23),))

eleven_bands_SED = np.array((temp_fuv, 
               gswlc_data['NUV'],
               gswlc_data['u'],
               gswlc_data['g'],
               gswlc_data['r'],
               gswlc_data['i'],
               gswlc_data['z'],
               gswlc_data['W1'],
               temp_w2,
               temp_w3,
               temp_w4,)).T

tempt50 = np.array(gswlc_data['t_50']).copy()
tempt50[(tempt50 > 13.7)] = 13.7

tempt10 = np.array(gswlc_data['t_90']).copy()
tempt10[(tempt10 > 13.7)] = 13.7

tempt90 = np.array(gswlc_data['t_90']).copy()
tempt90[(tempt90 > 13.7)] = 13.7

tempssfr = np.array(gswlc_data['logSSFR_50']).copy()
tempssfr[tempssfr < -3.5] = tempssfr[tempssfr < -3.5] + 0.3*np.random.normal(size=(np.sum(tempssfr < -3.5),))

tempmet = np.array(gswlc_data['logOH_50']).copy()
tempmet[tempmet < 7.1] = tempmet[tempmet < 7.1] + 0.1*np.random.normal(size=(np.sum(tempmet < 7.1),))

theta = np.vstack((gswlc_data['logM_50_x'], 
                   tempssfr,
                   gswlc_data['specz'],
                   tempt50,
                   gswlc_data['tauV_50'],
                   tempmet)).T
                #    manga_scalars['sersic_n'])).T
theta[np.isnan(theta)] = 0.

five_bands_SED= np.array((
               gswlc_data['u'],
               gswlc_data['g'],
               gswlc_data['r'],
               gswlc_data['i'],
               gswlc_data['z'],
               )).T

def safe_log_sfr(arr):
            return np.log10(np.clip(arr, 1e-3, None))

np.random.seed(256)
theta_full_sfh_sfr = np.vstack((gswlc_data['logM_50_x'], 
                   safe_log_sfr(timescales_data_cami_fixed['now']['sfr']),
                   gswlc_data['specz'],
                   tempt50,
                   gswlc_data['tauV_50'],
                   tempmet,
                   tempt10,
                   tempt90,
                  safe_log_sfr(timescales_data_cami_fixed[0.1]['sfr']),
                  safe_log_sfr(timescales_data_cami_fixed[0.3]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[0.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[0.7]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[0.8]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[1.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[1.3]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[1.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[1.7]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[2.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[2.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[3.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[3.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[4.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[4.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[5.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[5.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[6.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[6.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[7.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[7.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[8.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[8.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[9.0]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[9.5]['sfr']),
                   safe_log_sfr(timescales_data_cami_fixed[10.0]['sfr']))).T

theta_full_sfh_sfr[np.isnan(theta_full_sfh_sfr)] = 0.
theta_full_sfh_sfr[:,1][theta_full_sfh_sfr[:,1]<-2.5] = theta_full_sfh_sfr[:,1][theta_full_sfh_sfr[:,1]<-2.5] + 0.35*np.random.normal(size=(np.sum(theta_full_sfh_sfr[:,1]<-2.5),))
theta_full_sfh_sfr[:,8:][theta_full_sfh_sfr[:,8:]<-2.5] = theta_full_sfh_sfr[:,8:][theta_full_sfh_sfr[:,8:]<-2.5] + 0.3*np.random.normal(size=(np.sum(theta_full_sfh_sfr[:,8:]<-2.5),))



In [ ]:
#Importing trained ILI models 

#Image Models
posterior_ensemble_finetuned_vit, summaries_finetuned_vit = hickle.load('/data1/juanpabloalfonzo/ViT_Data/clean_trained_ILIs/posterior_ensemble_image_full_sfh_sfr_v2_full_train.hickle')
posterior_ensemble_half_size_im, summaries_half_size_im = hickle.load('/data1/juanpabloalfonzo/ViT_Data/clean_trained_ILIs/posterior_ensemble_half_size_image_full_train.hickle')
posterior_ensemble_quarter_size_im, summaries_quarter_size_im = hickle.load('/data1/juanpabloalfonzo/ViT_Data/clean_trained_ILIs/posterior_ensemble_quarter_size_image_full_train.hickle')
posterior_ensemble_g_band_im, summaries_g_band_im = hickle.load('/data1/juanpabloalfonzo/ViT_Data/clean_trained_ILIs/posterior_ensemble_g_band_image_full_train.hickle')

#Photometry Models
posterior_ensemble_sed_5_bands , summaries_sed_5_bands = hickle.load('/data1/juanpabloalfonzo/ViT_Data/clean_trained_ILIs/posterior_ensemble_5_band_full_sfh_sfr_v2_full_train.hickle')
posterior_ensemble_sed_11_bands , summaries_sed_11_bands = hickle.load('/data1/juanpabloalfonzo/ViT_Data/clean_trained_ILIs/posterior_ensemble_11_band_full_sfh_sfr_v2_full_train.hickle')


#Combined Models
posterior_ensemble_combined_11_bands, summaries_combined_11_bands = hickle.load('/data1/juanpabloalfonzo/ViT_Data/clean_trained_ILIs/posterior_ensemble_combined_11_band_full_sfh_sfr_v2_full_train.hickle')
posterior_ensemble_combined_5_bands_finetuned_vit, summaries_combined_5_bands_finetuned_vit = hickle.load('/data1/juanpabloalfonzo/ViT_Data/clean_trained_ILIs/posterior_ensemble_combined_5_band_full_sfh_sfr_v2_full_train.hickle')


#Importing ViT embeddings
gswlc_embeds_half_size = hickle.load('/data1/juanpabloalfonzo/ViT_Data/ViT_embeds/gswlc_emebds_half_size_clean.hickle')
gswlc_embeds_quarter_size = hickle.load('/data1/juanpabloalfonzo/ViT_Data/ViT_embeds/gswlc_emebds_quarter_size_clean.hickle')
gswlc_embeds_g_band = hickle.load('/data1/juanpabloalfonzo/ViT_Data/ViT_embeds/gswlc_emebds_g_band_clean.hickle')
gswlc_finetuned_embeds = hickle.load('/data1/juanpabloalfonzo/ViT_Data/ViT_embeds/finetuned_embeds.hickle')


# Extracting Predictions & Covariances (Demo Loop with Image Model)

In [ ]:
# all_covs_finetuned_vit_full, all_samples_finetuned_vit_full = [], []
# for gid in tqdm(range(len(gswlc_data))):
#     xobs_im = gswlc_finetuned_embeds[gid,:]
#     x_truth = theta[gid,:]

#     posterior_samples_finetuned_vit = posterior_ensemble_finetuned_vit.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars = False)
#     posterior_samples_finetuned_vit = posterior_samples_finetuned_vit.cpu().numpy()
#     covs_finetuned_vit, _ = check_degeneracy(posterior_samples_finetuned_vit)

#     all_covs_finetuned_vit_full.append(covs_finetuned_vit)
#     all_samples_finetuned_vit_full.append(posterior_samples_finetuned_vit)

# all_covs_finetuned_vit_full = np.array(all_covs_finetuned_vit_full)
# all_samples_finetuned_vit_full = np.array(all_samples_finetuned_vit_full)

# hickle.dump((all_covs_finetuned_vit_full, all_samples_finetuned_vit_full), '/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/all_covs_and_samples_finetuned_vit_full.hickle')


# Importing Predictions from Disk 
Warning: Will use a LOT of RAM. Only load samples and covs for the model you are interested in exploring

In [ ]:
#Image Models 
all_covs_image, all_samples_image = hickle.load('/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/all_samples_and_covs_image_full_sfh_sfr_v2_full_sample.hickle')
all_covs_image = np.array(all_covs_image)
all_samples_image = np.array(all_samples_image)

all_covs_half_size_im, all_samples_half_size_im = hickle.load('/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/half_size_image.hickle')
all_covs_half_size_im = np.array(all_covs_half_size_im)
all_samples_half_size_im = np.array(all_samples_half_size_im)

all_covs_quarter_size_im, all_samples_quarter_size_im = hickle.load('/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/quarter_size_image.hickle')
all_covs_quarter_size_im = np.array(all_covs_quarter_size_im)
all_samples_quarter_size_im = np.array(all_samples_quarter_size_im)

all_g_band_covs_im, all_g_band_samples_im = hickle.load('/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/g_band_image.hickle')
all_g_band_covs_im = np.array(all_g_band_covs_im)
all_g_band_samples_im = np.array(all_g_band_samples_im)



#Photometry Models 
all_covs_5_band, all_samples_5_band = hickle.load('/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/5_band_sfh_v2.hickle')
all_covs_5_band = np.array(all_covs_5_band)
all_samples_5_band = np.array(all_samples_5_band)

all_covs_11_band, all_samples_11_band = hickle.load('/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/11_band_sfh_v2.hickle')
all_covs_11_band = np.array(all_covs_11_band)
all_samples_11_band = np.array(all_samples_11_band)

#Combined Models (make all "bands" to "band")
all_covs_combined_5_band, all_samples_combined_5_band = hickle.load('/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/combined_5_band_sfh_v2.hickle')
all_covs_combined_5_band = np.array(all_covs_combined_5_band)
all_samples_combined_5_band = np.array(all_samples_combined_5_band)

all_covs_combined_11_band, all_samples_combined_11_band = hickle.load('/data1/juanpabloalfonzo/ViT_Data/samples_and_covs/combined_11_band_sfh_v2.hickle')
all_covs_combined_11_band = np.array(all_covs_combined_11_band)
all_samples_combined_11_band = np.array(all_samples_combined_11_band)

In [ ]:
mvals_full = theta[:,0]
sfrvals_full = theta[:,1] + theta[:,0] - 9

# Calculating Prediction Errors, 1$\sigma$ Intervals & Population Metrics
Note: Since all models predict the 6 physical parameters alongside the SFHs (34 dimensions total), we slice `[:, :, 0:6]` when evaluating physical parameter predictions and `[:, 0:5, 0:5]` when evaluating their $5\times 5$ lower-triangular covariance matrices.

In [ ]:
# Load re-interpolated true SFHs for SFH reconstruction plots
gswlc_sfh_load = np.load('/data1/juanpabloalfonzo/ViT_Data/GSWLC_cami_sfhs_reinterp.npz')
gswlc_sfh_mod = np.array([gswlc_sfh_load['sfhs'], gswlc_sfh_load['timeax']]).transpose(1, 0, 2) # Shape: (N_galaxies, 2, N_timepoints)

# Test set slices for convenience
mvals = mvals_full[train_index:]
sfrvals = sfrvals_full[train_index:]

In [ ]:
# Calculate absolute prediction errors on the 6 physical parameters ([:, :, 0:6])
# Used for Dusty Starburst analysis (Figure 9) and UMAP residuals (Figure 8)
n_eval_5b = len(all_samples_5_band)
truth_eval_5b = theta if n_eval_5b == len(theta) else theta[train_index:]
samples_im_eval_5b = all_samples_image[:, :, 0:6] if n_eval_5b == len(theta) else all_samples_image[train_index:, :, 0:6]

allimerr_full_new = np.abs(np.nanmean(samples_im_eval_5b, axis=1) - truth_eval_5b)
allsederr_5_bands_full_new = np.abs(np.nanmean(all_samples_5_band[:, :, 0:6], axis=1) - truth_eval_5b)
allsederr_combined_5_bands_finetuned_vit_full_new = np.abs(np.nanmean(all_samples_combined_5_band[:, :, 0:6], axis=1) - truth_eval_5b)

In [ ]:
# Calculate 1-Sigma Credible Intervals ((P84 - P16) / 2) for the 6 physical parameters
param_names = ['log M*', 'log sSFR', 'z', 't50', 'Av', '12+log(O/H)']

primary_models_6d = {
    "Finetuned ViT": all_samples_image[:, :, 0:6],
    "5 Band Photometry": all_samples_5_band[:, :, 0:6],
    "Combined (ViT + Bands)": all_samples_combined_5_band[:, :, 0:6]
}

primary_individual_scatters = {}
for model_name, samples_6d in primary_models_6d.items():
    p16 = np.nanpercentile(samples_6d, 16, axis=1)
    p84 = np.nanpercentile(samples_6d, 84, axis=1)
    primary_individual_scatters[model_name] = (p84 - p16) / 2.0
    print(f"Saved individual scatters for {model_name:<25} | Shape: {primary_individual_scatters[model_name].shape}")

models_11_bands_6d = {
    "11 Band Photometry": all_samples_11_band[:, :, 0:6],
    "11 Band Combined": all_samples_combined_11_band[:, :, 0:6]
}

primary_individual_scatters_11_bands = {}
for model_name, samples_6d in models_11_bands_6d.items():
    p16 = np.nanpercentile(samples_6d, 16, axis=1)
    p84 = np.nanpercentile(samples_6d, 84, axis=1)
    primary_individual_scatters_11_bands[model_name] = (p84 - p16) / 2.0
    print(f"Saved individual scatters for {model_name:<25} | Shape: {primary_individual_scatters_11_bands[model_name].shape}")

# Tables 1 & 2: Population Bias ($\mu$) and Scatter ($\sigma_{\mathrm{NMAD}}$)

In [ ]:
def calculate_population_metrics(samples, truth):
    """
    Calculates Bias (median residual), NMAD Scatter, and 68% Coverage for the 6 physical parameters.
    samples shape: (N_gal, N_samples, 6)
    truth shape:   (N_gal, 6)
    """
    p16 = np.nanpercentile(samples, 16, axis=1)
    p50 = np.nanpercentile(samples, 50, axis=1)
    p84 = np.nanpercentile(samples, 84, axis=1)
    
    residuals = p50 - truth
    bias = np.nanmedian(residuals, axis=0)
    nmad = 1.4826 * np.nanmedian(np.abs(residuals - bias), axis=0)
    
    is_covered = (truth >= p16) & (truth <= p84)
    coverage_pct = np.nanmean(is_covered, axis=0) * 100
    
    return bias, nmad, coverage_pct

# --- TABLE 1: Imaging vs 5-Band Photometry vs Combined ---
print("=== TABLE 1: Primary Models ===")
print(f"{'Model':<25} | {'Parameter':<12} | {'Bias':<8} | {'Scatter':<8} | {'Coverage'}")
print("-" * 75)

table1_models = {
    "Imaging": all_samples_image[:, :, 0:6],
    "5-band Photometry": all_samples_5_band[:, :, 0:6],
    "Combined": all_samples_combined_5_band[:, :, 0:6]
}

for model_name, samples_array in table1_models.items():
    truth_to_use = theta if len(samples_array) == len(theta) else theta[train_index:]
    bias_arr, scatter_arr, cov_arr = calculate_population_metrics(samples_array, truth_to_use)
    for i, param in enumerate(param_names):
        print(f"{model_name:<25} | {param:<12} | {bias_arr[i]:>8.3f} | {scatter_arr[i]:>8.3f} | {cov_arr[i]:>5.1f}%")
    print("-" * 75)

In [ ]:
# --- TABLE 2: Image Ablation Models (gri vs g-band vs Half-Size vs Quarter-Size) ---
print("=== TABLE 2: Image Ablation Models ===")
print(f"{'Model':<25} | {'Parameter':<12} | {'Bias':<8} | {'Scatter':<8}")
print("-" * 60)

table2_models = {
    "gri Image": all_samples_image[:, :, 0:6],
    "g-Band Image": all_g_band_samples_im[:, :, 0:6],
    "Half-Size Image": all_samples_half_size_im[:, :, 0:6],
    "Quarter-Size Image": all_samples_quarter_size_im[:, :, 0:6]
}

ablation_metrics = {}
for model_name, samples_array in table2_models.items():
    ablation_metrics[model_name] = {}
    truth_to_use = theta if len(samples_array) == len(theta) else theta[train_index:]
    bias_arr, scatter_arr, _ = calculate_population_metrics(samples_array, truth_to_use)
    for i, param in enumerate(param_names):
        ablation_metrics[model_name][param] = {'bias': bias_arr[i], 'scatter': scatter_arr[i]}
        print(f"{model_name:<25} | {param:<12} | {bias_arr[i]:>8.3f} | {scatter_arr[i]:>8.3f}")
    print("-" * 60)

In [ ]:
# --- 11-Band Photometry & Combined Metrics (Appendix C / D) ---
print("=== 11-Band Models ===")
print(f"{'Model':<25} | {'Parameter':<12} | {'Bias':<8} | {'Scatter':<8} | {'Coverage'}")
print("-" * 75)

table_11b_models = {
    "Finetuned ViT": all_samples_image[:, :, 0:6],
    "11 Band Photometry": all_samples_11_band[:, :, 0:6],
    "11 Band Combined": all_samples_combined_11_band[:, :, 0:6]
}

all_metrics_11_bands = {}
for model_name, samples_array in table_11b_models.items():
    all_metrics_11_bands[model_name] = {}
    truth_to_use = theta if len(samples_array) == len(theta) else theta[train_index:]
    bias_arr, scatter_arr, cov_arr = calculate_population_metrics(samples_array, truth_to_use)
    for i, param in enumerate(param_names):
        all_metrics_11_bands[model_name][param] = {'bias': bias_arr[i], 'scatter': scatter_arr[i], 'coverage': cov_arr[i]}
        print(f"{model_name:<25} | {param:<12} | {bias_arr[i]:>8.3f} | {scatter_arr[i]:>8.3f} | {cov_arr[i]:>5.1f}%")
    print("-" * 75)

# Figure 1: Distribution of Dataset in $M_*$-SFR Space

In [ ]:
np.random.seed(42)
random_gals_sample = np.random.choice(len(gswlc_data), 100, replace=False)
plt.figure(figsize=(10, 8))
plt.hist2d(gswlc_data['logM_50_x'], gswlc_data['logSFR_50'], bins=30, cmap='Blues', vmax=2500)
plt.errorbar(gswlc_data['logM_50_x'][random_gals_sample], gswlc_data['logSFR_50'][random_gals_sample], 
             yerr=(gswlc_data['logSFR_84'][random_gals_sample]-gswlc_data['logSFR_50'][random_gals_sample],
                   gswlc_data['logSFR_50'][random_gals_sample]-gswlc_data['logSFR_16'][random_gals_sample]), 
             xerr=(gswlc_data['logM_84'][random_gals_sample]-gswlc_data['logM_50_x'][random_gals_sample],
                   gswlc_data['logM_50_x'][random_gals_sample]-gswlc_data['logM_16'][random_gals_sample]), 
             marker='.', markersize=3, alpha=0.4, lw=0, elinewidth=2, capsize=3, color='black', label='100 Random Galaxies')
plt.ylim(-3.1, 3)
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
plt.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')
plt.colorbar(label='Number Density')
plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18)
plt.ylabel('log SFR [$M_{\odot}$/yr]', fontsize=18)
plt.title('Distribution of Dataset in $M_*$-SFR Space', fontsize=20)
plt.legend(fontsize=14)
# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/num_density.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Figure 3: Inferred Posteriors (5-Band Photometry vs. Image vs. Combined)

In [ ]:
import matplotlib.patches as mpatches

clrs = plt.cm.Spectral(np.linspace(0, 1, 10))
ranges = [(9, 12), (-4.5, 3), (0, 0.16), (0, 13.7), (0, 3), (6.5, 9.5)]
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

np.random.seed(256)
random_gal = np.random.randint(train_index, len(gswlc_data), size=(1))
gid = int(random_gal[0])

xobs_sed = five_bands_SED[gid, :]
xobs_im = gswlc_finetuned_embeds[gid, :]
x_truth = theta[gid, :]

# Sample from models and slice [:, 0:6] for the 6 physical parameters
# posterior_samples_sed = posterior_ensemble_sed_5_bands.sample((1000,), torch.Tensor(xobs_sed).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
# posterior_samples_im = posterior_ensemble_finetuned_vit.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
# posterior_samples_combined = posterior_ensemble_combined_5_bands_finetuned_vit.sample((1000,), torch.Tensor(np.hstack((xobs_im, xobs_sed))).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]

posterior_samples_sed = all_samples_5_band[gid, :, 0:6]
posterior_samples_im = all_samples_image[gid, :, 0:6]
posterior_samples_combined = all_samples_combined_5_band[gid, :, 0:6]

plt.imshow(Image.open(io.BytesIO(gswlc_images_skyserver[gid])))
plt.axis('off')
plt.figure()
plt.scatter(np.arange(5), xobs_sed)
plt.xticks(np.arange(0, 5), ['u', 'g', 'r', 'i', 'z'], fontsize=18)
plt.xlabel('Band', fontsize=18)
plt.ylabel('Magnitude', fontsize=18)

fig = corner.corner(posterior_samples_sed, truths=x_truth, color='dodgerblue', range=ranges, labels=labels, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])
fig = corner.corner(posterior_samples_im, truths=x_truth, truth_color='k', color='firebrick', range=ranges, labels=labels, fig=fig, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])
fig = corner.corner(posterior_samples_combined, truths=x_truth, truth_color='k', color='green', range=ranges, labels=labels, fig=fig, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])

sed_legend_patch = mpatches.Patch(color='dodgerblue', label='Photometry')
image_legend_patch = mpatches.Patch(color='firebrick', label='Image')
combined_legend_patch = mpatches.Patch(color='green', label='Combined')
fig.legend(handles=[sed_legend_patch, image_legend_patch, combined_legend_patch],
           loc='upper right', fontsize='18', title='Posterior Source', frameon=True, title_fontsize='18')
fig.set_size_inches(12, 12)
for label in fig.get_axes():
    label.xaxis.label.set_size(18)
    label.yaxis.label.set_size(18)
    for tick in label.get_xticklabels():
        tick.set_fontsize(14)
    for tick in label.get_yticklabels():
        tick.set_fontsize(14)
plt.show()

# Figure 4: 5-Band Photometry vs. Image vs. Combined $1\sigma$ Credible Interval

In [ ]:
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

# Match coordinate length to sample array length
n_plot_5b = len(primary_individual_scatters["5 Band Photometry"])
x_m_5b = mvals_full if n_plot_5b == len(mvals_full) else mvals
y_s_5b = sfrvals_full if n_plot_5b == len(sfrvals_full) else sfrvals
vit_scatters_5b = primary_individual_scatters["Finetuned ViT"] if len(primary_individual_scatters["Finetuned ViT"]) == n_plot_5b else primary_individual_scatters["Finetuned ViT"][train_index:]

plt.subplots(figsize=(15, 21), nrows=6, ncols=3)
plt.suptitle('5 Band Photometry vs Image vs Combined 1$\sigma$ Credible Interval', fontsize=18, y=1)
for qty in range(6):
    c_lower_bound = np.nanpercentile(vit_scatters_5b[:, qty], 2)
    c_upper_bound = np.nanpercentile(vit_scatters_5b[:, qty], 98)
    
    plt.subplot(6, 3, qty*3+1)
    plt.hexbin(x_m_5b, y_s_5b, gridsize=20, mincnt=2, C=primary_individual_scatters["5 Band Photometry"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18) 
    plt.ylabel('log SFR  [$M_{\odot}$/yr]', fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$ ' + labels[qty] + '$_{,Photometry}$', fontsize=18)
    
    plt.subplot(6, 3, qty*3+2)
    plt.hexbin(x_m_5b, y_s_5b, gridsize=20, mincnt=2, C=vit_scatters_5b[:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$  [$M_{\odot}$]', fontsize=18)  
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$ ' + labels[qty] + '$_{,Image}$', fontsize=18)
    plt.tight_layout()

    plt.subplot(6, 3, qty*3+3)
    plt.hexbin(x_m_5b, y_s_5b, gridsize=20, mincnt=2, C=primary_individual_scatters["Combined (ViT + Bands)"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$  [$M_{\odot}$]', fontsize=18) 
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$ ' + labels[qty] + '$_{,Photometry+Image}$', fontsize=18)

# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/5_bands_SED_vs_image_vs_combined_finetuned.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Figure 5: Population-Level Degeneracies ($t_{50}$, $A_v$, $12+\log(\mathrm{O/H})$ Residuals)

In [ ]:
import matplotlib.lines as mlines

FAST_MODE = False  
STRIDE = 50       

def get_pooled_residuals(samples_array):
    medians = np.nanmedian(samples_array, axis=1, keepdims=True)
    residuals = samples_array - medians
    return residuals.reshape(-1, samples_array.shape[-1])

# 1. Compute pooled residuals on the 6 physical parameters ([:, :, 0:6])
subset_bands = get_pooled_residuals(all_samples_5_band[:, :, 0:6])
subset_combined = get_pooled_residuals(all_samples_image[:, :, 0:6])

if FAST_MODE:
    print(f"Running in FAST_MODE. Plotting 1/{STRIDE} of the samples...")
    subset_bands = subset_bands[::STRIDE]
    subset_combined = subset_combined[::STRIDE]

# 2. Extract Pre-calculated Median Covariances for the 6 physical parameters ([:, 0:5, 0:5])
median_cov_bands = np.nanmedian(all_covs_5_band[:, 0:5, 0:5], axis=0)
median_cov_combined = np.nanmedian(all_covs_image[:, 0:5, 0:5], axis=0)

# 3. Setup plotting parameters for [t50, Av, 12+log(O/H)]
indices_to_plot = [3, 4, 5]
n_params_subset = len(indices_to_plot)

subset_bands = subset_bands[:, indices_to_plot]
subset_combined = subset_combined[:, indices_to_plot]

subset_labels = [r'$\Delta$ t$_{50}$', r'$\Delta$ Av', r'$\Delta$ 12+log(O/H)'] 
color_bands = 'dodgerblue'
color_combined = 'red'
plot_range = [(-6, 6), (-2.5, 2.5), (-1.5, 1.5)] 

kwargs_bands = {
    "labels": subset_labels,
    "label_kwargs": {"fontsize": 18},
    "hist_kwargs": {"density": True, "linewidth": 2},
    "plot_datapoints": False,
    "plot_density": True,
    "fill_contours": True,
    "alpha": 0.3,                 
    "levels": (0.68, 0.95),
    "range": plot_range,          
    "bins": 30,                   
}

kwargs_combined = {
    "labels": subset_labels,
    "hist_kwargs": {"density": True, "linewidth": 2.5},
    "plot_datapoints": False,
    "plot_density": False,        
    "fill_contours": False,       
    "contour_kwargs": {"linewidths": 2.5}, 
    "levels": (0.68, 0.95),
    "range": plot_range,          
    "bins": 30,
}

# 4. Generate the corner plot
fig = plt.figure(figsize=(12, 12))
fig = corner.corner(subset_bands, fig=fig, color=color_bands, **kwargs_bands)
corner.corner(subset_combined, fig=fig, color=color_combined, **kwargs_combined)

axes = np.array(fig.axes).reshape((n_params_subset, n_params_subset))
bbox_props = dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.8)

for yi in range(n_params_subset):
    for xi in range(yi):
        ax = axes[yi, xi]
        ax.axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.axhline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.plot(0, 0, marker="s", color="black", markersize=3, zorder=5)
        
        orig_y = indices_to_plot[yi]
        orig_x = indices_to_plot[xi]
        
        rho_bands = median_cov_bands[orig_y - 1, orig_x]
        rho_comb = median_cov_combined[orig_y - 1, orig_x]
        
        txt_bands = f"$\\rho_{{phot}} = {rho_bands:.2f}$"
        txt_comb = f"$\\rho_{{img}} = {rho_comb:.2f}$"
        
        ax.text(0.05, 0.96, txt_bands, transform=ax.transAxes, 
                fontsize=16, color=color_bands, ha='left', va='top', weight='bold', bbox=bbox_props)
        ax.text(0.05, 0.86, txt_comb, transform=ax.transAxes, 
                fontsize=16, color=color_combined, ha='left', va='top', weight='bold', bbox=bbox_props)
    
    axes[yi, yi].axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)

legend_handles = [
    mlines.Line2D([], [], color=color_bands, lw=4, label='5 Band Photometry'),
    mlines.Line2D([], [], color=color_combined, lw=3, label='Image')
]
fig.axes[n_params_subset - 1].legend(handles=legend_handles, loc='center left', fontsize=18, frameon=False)
plt.suptitle("Population-Level Degeneracies (Residuals)", fontsize=22, y=1.02)
# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/population_corner.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Figure 6: 10 Gyr SFH Reconstruction Across $M_*$-sSFR Bins
In the 34-dimensional target array `theta_full_sfh_sfr`, index `1` corresponds to present-day $\log\mathrm{SFR}$, indices `6` and `7` are $t_{10}$ and $t_{90}$, and indices `8` through `33` correspond to $\log\mathrm{SFR}$ at the 26 lookback times from $0.1$ to $10.0\,\mathrm{Gyr}$.

In [ ]:
# Mapping of lookback time to column index in the 34D sample arrays
time_to_idx_34d = {
    0.0: 1, 0.1: 8, 0.3: 9, 0.5: 10, 0.7: 11, 0.8: 12, 1.0: 13,
    1.3: 14, 1.5: 15, 1.7: 16, 2.0: 17, 2.5: 18, 3.0: 19,
    3.5: 20, 4.0: 21, 4.5: 22, 5.0: 23, 5.5: 24, 6.0: 25,
    6.5: 26, 7.0: 27, 7.5: 28, 8.0: 29, 8.5: 30, 9.0: 31,
    9.5: 32, 10.0: 33
}

timescale_indices_34d = {
    ("today" if t == 0.0 else f"{t}gyr"): idx for t, idx in time_to_idx_34d.items()
}

# Build dictionary of median predicted log SFR for each lookback time from all_samples_image
n_im_samples = len(all_samples_image)
slice_im = slice(None) if n_im_samples == len(mvals_full) else slice(train_index, None)

predicted_timescales_image = {}
for t, col_idx in time_to_idx_34d.items():
    key = "now" if t == 0.0 else f"{t}gyr"
    t_key = "now" if t == 0.0 else t
    predicted_timescales_image[key] = (
        timescales_data_cami_fixed[t_key]['mass'][slice_im],
        np.nanmedian(all_samples_image[:, :, col_idx], axis=1)
    )

In [ ]:
reconstruction_times = [
    0.0, 0.1, 0.3, 0.5, 0.7, 0.8, 1.0, 1.3, 1.5, 1.7, 2.0,
    2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 
    7.5, 8.0, 8.5, 9.0, 9.5, 10.0
]

mass_bins = [9.0, 10.2, 10.8, 12.0]
mass_labels = ["Low Mass (<10.2)", "Med Mass (10.2-10.8)", "High Mass (>10.8)"]

ssfr_bins = [-13.0, -11.5, -10.5, -9.5, -7.0] 
ssfr_labels = [
    "Quiescent (sSFR < -11.5)", 
    "Green Valley(-11.5 < sSFR < -10.5)", 
    "Main Sequence (-10.5 < sSFR < -9.5)", 
    "Starburst (sSFR > -9.5)"
]

gal_mass = mvals_full[slice_im]
gal_sfr  = sfrvals_full[slice_im]
gal_ssfr = gal_sfr - gal_mass
sfh_mod_sub = gswlc_sfh_mod[slice_im]

fig, axes = plt.subplots(4, 3, figsize=(18, 16), sharex=True, sharey=False)
common_lookback_grid = np.linspace(0, 10.0, 200)

print("Generating 10 Gyr Reconstruction Grid (4 Rows: SB, MS, GV, Q)...")

for row_idx, ssfr_idx in enumerate([3, 2, 1, 0]): 
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]
        
        m_min, m_max = mass_bins[col_idx], mass_bins[col_idx+1]
        s_min, s_max = ssfr_bins[ssfr_idx], ssfr_bins[ssfr_idx+1]
        
        bin_mask = (gal_mass >= m_min) & (gal_mass < m_max) & \
                   (gal_ssfr >= s_min) & (gal_ssfr < s_max)
        
        indices = np.where(bin_mask)[0]
        n_count = len(indices)
        bin_title = f"{mass_labels[col_idx]}\n{ssfr_labels[ssfr_idx]}\n(N={n_count})"

        if n_count == 0:
            ax.text(0.5, 0.5, "No Galaxies", ha='center', transform=ax.transAxes)
            ax.set_title(bin_title, fontsize=10)
            continue

        true_sfhs_list = []
        pred_sfhs_list = []
        
        for idx in indices:
            sfr_true = sfh_mod_sub[idx, 0, :]  
            t_cosmic = sfh_mod_sub[idx, 1, :]
            t_obs = np.max(t_cosmic) 
            t_lookback_gal = t_obs - t_cosmic
            
            sort_order = np.argsort(t_lookback_gal)
            sfr_interp = np.interp(common_lookback_grid, 
                                   t_lookback_gal[sort_order], 
                                   sfr_true[sort_order])
            true_sfhs_list.append(sfr_interp)
            
            preds = []
            for t in reconstruction_times:
                key = 'now' if t == 0.0 else f'{t}gyr'
                preds.append(predicted_timescales_image[key][1][idx])
            pred_sfhs_list.append(preds)

        true_arr = np.array(true_sfhs_list)
        pred_arr = np.array(pred_sfhs_list)
            
        true_med = np.nanmedian(true_arr, axis=0)
        true_p16 = np.nanpercentile(true_arr, 16, axis=0)
        true_p84 = np.nanpercentile(true_arr, 84, axis=0)
        
        pred_med = np.nanmedian(pred_arr, axis=0)
        pred_p16 = np.nanpercentile(pred_arr, 16, axis=0)
        pred_p84 = np.nanpercentile(pred_arr, 84, axis=0)

        ax.fill_between(common_lookback_grid, true_p16, true_p84, 
                        color='black', alpha=0.1, label='True Spread')
        ax.plot(common_lookback_grid, true_med, color='black', lw=2, label='Avg Truth')
        
        ax.fill_between(reconstruction_times, 10**pred_p16, 10**pred_p84, 
                        color='red', alpha=0.2, label='Model Spread')
        ax.plot(reconstruction_times, 10**pred_med, color='red', 
                linestyle='--', marker='o', markersize=3, label='Avg Model')

        ax.set_title(bin_title, fontsize=14)
        ax.set_xlim(-0.2, 10.2)

        if col_idx == 0: ax.set_ylim(0, 3)
        if col_idx == 1: ax.set_ylim(0, 12)
        if col_idx == 2: ax.set_ylim(0, 30)

        ax.grid(alpha=0.3)
        if row_idx == 0 and col_idx == 0:
            ax.legend(loc='upper left', fontsize=14)

fig.supxlabel("Lookback Time [Gyr] (0 = Today)", fontsize=16)
fig.supylabel("SFR [$M_\odot$/yr]", fontsize=16)
plt.tight_layout()
# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/10Gyr_reconstruction_grid_not_log.pdf', dpi=300)
plt.show()

# Figure 7: Continuously Accurate SFH Prediction (Continuous Morphological Memory)

In [ ]:
def compute_sfh_memory_dicts(samples_34d, truth_34d):
    """
    Computes individual, cumulative (continuous), and step-failure accuracy dictionaries
    across the 27 SFH lookback timesteps in the 34D prediction array.
    """
    individual_acc = {}
    cumulative_acc = {}
    failed_step = {}
    current_intersection = None

    print(f"{'Time':<8} | {'Indiv Acc':<10} | {'Cumulative':<10} | {'Failed Here'}")
    print("-" * 45)

    for label, col_idx in timescale_indices_34d.items():
        samples = samples_34d[:, :, col_idx]
        p16 = np.nanpercentile(samples, 16, axis=1)
        p84 = np.nanpercentile(samples, 84, axis=1)
        truth = truth_34d[:, col_idx]
        
        with np.errstate(invalid='ignore'):
            is_accurate_mask = (truth >= p16) & (truth <= p84)
            
        acc_indices = np.where(is_accurate_mask)[0]
        individual_acc[label] = acc_indices

        if current_intersection is None:
            current_intersection = acc_indices
            failed_step[label] = np.array([])
        else:
            failures = np.setdiff1d(current_intersection, acc_indices)
            failed_step[label] = failures
            current_intersection = np.intersect1d(current_intersection, acc_indices)
            
        cumulative_acc[label] = current_intersection
        print(f"{label:<8} | {len(acc_indices):<10} | {len(current_intersection):<10} | {len(failed_step[label])}")
        
    return individual_acc, cumulative_acc, failed_step

truth_im_34d = theta_full_sfh_sfr if len(all_samples_image) == len(theta_full_sfh_sfr) else theta_full_sfh_sfr[train_index:]
individual_accurate_cami, cumulative_accurate_cami, failed_at_step_cami = compute_sfh_memory_dicts(all_samples_image, truth_im_34d)

In [ ]:
# Construct Continuous Memory Depth Array & Plot Figure 7
timescale_map = [
    (0.0, "today"), (0.1, "0.1gyr"), (0.3, "0.3gyr"), (0.5, "0.5gyr"),
    (0.7, "0.7gyr"), (0.8, "0.8gyr"), (1.0, "1.0gyr"), (1.3, "1.3gyr"),
    (1.5, "1.5gyr"), (1.7, "1.7gyr"), (2.0, "2.0gyr"), (2.5, "2.5gyr"),
    (3.0, "3.0gyr"), (3.5, "3.5gyr"), (4.0, "4.0gyr"), (4.5, "4.5gyr"),
    (5.0, "5.0gyr"), (5.5, "5.5gyr"), (6.0, "6.0gyr"), (6.5, "6.5gyr"),
    (7.0, "7.0gyr"), (7.5, "7.5gyr"), (8.0, "8.0gyr"), (8.5, "8.5gyr"),
    (9.0, "9.0gyr"), (9.5, "9.5gyr"), (10.0, "10.0gyr")
]

x_mass_im = mvals_full if len(all_samples_image) == len(mvals_full) else mvals
y_sfr_im  = sfrvals_full if len(all_samples_image) == len(sfrvals_full) else sfrvals

memory_depth = np.zeros(len(x_mass_im))
for t_val, key in timescale_map:
    if key in cumulative_accurate_cami:
        memory_depth[cumulative_accurate_cami[key]] = t_val

fig, ax = plt.subplots(figsize=(10, 8))
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
ax.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')

hb = ax.hexbin(x_mass_im, y_sfr_im, C=memory_depth, 
               gridsize=30, cmap='Spectral_r', reduce_C_function=np.mean,
               mincnt=5, vmin=0.0, vmax=8, edgecolors='none')

cb = fig.colorbar(hb, ax=ax, extend='both')
cb.set_label('Lookback Time [Gyr]', fontsize=18) 

ax.set_xlabel(r'$\log M_* \ [M_\odot]$', fontsize=18)
ax.set_ylabel(r'$\log \mathrm{SFR} \ [M_\odot/\mathrm{yr}]$', fontsize=18)
ax.set_title('Continuously Accurate SFH Prediction', fontsize=18)
ax.legend(loc='lower right')
ax.set_ylim(-5, 3.5)
ax.set_xlim(7.8, 12)
# fig.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/morphological_memory_depth_CI_flipped_colours_v2.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation check between continuous morphological memory and morphological summary statistics (Sec. 4.3)
morph_slice = slice(None) if len(memory_depth) == len(gswlc_data_merged) else slice(train_index, None)
for col_name in ['bulge_n_morph', 'ttype_morph', 'bt_morph', 'pbulge_morph']:
    y_pred = np.array(gswlc_data_merged[col_name])[morph_slice]
    mask = ~np.isnan(memory_depth) & ~np.isnan(y_pred)
    rho, p_val = spearmanr(memory_depth[mask], y_pred[mask])
    print(f"{col_name:<15} | Spearman rho: {rho:>7.4f} | p-value: {p_val:.4e}")

# Figure 8: UMAP Projection of the ViT Image Embedding Space

In [ ]:
import umap
import matplotlib.gridspec as gridspec

data_embeds = np.array(gswlc_finetuned_embeds)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean')
embedding = reducer.fit_transform(data_embeds)

In [ ]:
# Figure 8 (Lower Panels): 2x4 Grid Colored by Morphological, Physical, and Precision Metrics
fig, axes = plt.subplots(2, 4, figsize=(28, 10), sharex=True, sharey=True)

axis_label_fontsize = 18
cbar_label_fontsize = 18
cbar_tick_fontsize = 14 

fig.supxlabel("UMAP Dimension 1", fontsize=axis_label_fontsize)
fig.supylabel("UMAP Dimension 2", fontsize=axis_label_fontsize, x=-0.01)

# --- Subplot 1 (Top, Col 1): Prominent Bulge ---
ax1 = axes[0, 0]
data1 = gswlc_data_merged['pbulge_morph']
hb1 = ax1.hexbin(embedding[:, 0], embedding[:, 1], C=data1, gridsize=40, cmap='magma',
                 vmin=np.nanpercentile(data1, 2), vmax=np.nanpercentile(data1, 98))
cbar1 = fig.colorbar(hb1, ax=ax1)
cbar1.set_label('Prominent Bulge', fontsize=cbar_label_fontsize)
cbar1.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 3 (Bottom, Col 1): Av ---
ax3 = axes[1, 0]
data3 = gswlc_data_merged['tauV_50']
hb3 = ax3.hexbin(embedding[:, 0], embedding[:, 1], C=data3, gridsize=40, cmap='magma',
                 vmin=np.nanpercentile(data3, 2), vmax=np.nanpercentile(data3, 98))
cbar3 = fig.colorbar(hb3, ax=ax3)
cbar3.set_label('Av', fontsize=cbar_label_fontsize)
cbar3.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 2 (Top, Col 2): T-Type ---
ax2 = axes[0, 1]
data2 = gswlc_data_merged['ttype_morph']
hb2 = ax2.hexbin(embedding[:, 0], embedding[:, 1], C=data2, gridsize=40, cmap='magma',
                 vmin=np.nanpercentile(data2, 2), vmax=np.nanpercentile(data2, 98))
cbar2 = fig.colorbar(hb2, ax=ax2)
cbar2.set_label('T-Type', fontsize=cbar_label_fontsize)
cbar2.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 4 (Bottom, Col 2): sSFR ---
ax4 = axes[1, 1]
data4 = gswlc_data_merged['logSSFR_50'] - 9
hb4 = ax4.hexbin(embedding[:, 0], embedding[:, 1], C=data4, gridsize=40, cmap='magma',
                 vmin=np.nanpercentile(data4, 2), vmax=np.nanpercentile(data4, 98))
cbar4 = fig.colorbar(hb4, ax=ax4)
cbar4.set_label('sSFR', fontsize=cbar_label_fontsize)
cbar4.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 5 (Top, Col 3): Continuous Accurate SFH Prediction ---
ax5 = axes[0, 2]
emb_im = embedding if len(memory_depth) == len(embedding) else embedding[train_index:]
data5 = memory_depth
hb5 = ax5.hexbin(emb_im[:, 0], emb_im[:, 1], C=data5, gridsize=40, cmap='Spectral_r',
                 vmin=0, vmax=8)
cbar5 = fig.colorbar(hb5, ax=ax5, extend='both')
cbar5.set_label('Continuous Accurate SFH Prediction', fontsize=cbar_label_fontsize)
cbar5.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 6 (Bottom, Col 3): sSFR Prediction Residuals ---
ax6 = axes[1, 2]
emb_5b = embedding if len(allsederr_combined_5_bands_finetuned_vit_full_new) == len(embedding) else embedding[train_index:]
data6 = allsederr_combined_5_bands_finetuned_vit_full_new[:, 1]
hb6 = ax6.hexbin(emb_5b[:, 0], emb_5b[:, 1], C=data6, gridsize=40, cmap='viridis',
                 vmin=np.nanpercentile(data6, 2), vmax=np.nanpercentile(data6, 98))
cbar6 = fig.colorbar(hb6, ax=ax6)
cbar6.set_label('sSFR Prediction Residuals', fontsize=cbar_label_fontsize)
cbar6.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 7 (Top, Col 4): sSFR Inferred Posterior Width ---
ax7 = axes[0, 3]
data7 = primary_individual_scatters["Finetuned ViT"][:, 1]
hb7 = ax7.hexbin(emb_im[:, 0], emb_im[:, 1], C=data7, gridsize=40, cmap='viridis',
                 vmin=np.nanpercentile(data7, 2), vmax=np.nanpercentile(data7, 98))
cbar7 = fig.colorbar(hb7, ax=ax7)
cbar7.set_label('sSFR Inferred Posterior Width', fontsize=cbar_label_fontsize)
cbar7.ax.tick_params(labelsize=cbar_tick_fontsize)

# --- Subplot 8 (Bottom, Col 4): Number Density ---
ax8 = axes[1, 3]
hb8 = ax8.hexbin(embedding[:, 0], embedding[:, 1], gridsize=40, cmap='Blues', mincnt=1)
cbar8 = fig.colorbar(hb8, ax=ax8)
cbar8.set_label('Number Density', fontsize=cbar_label_fontsize)
cbar8.ax.tick_params(labelsize=cbar_tick_fontsize)

plt.tight_layout()  
# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/umap_vit_embeddings_finetuned_extended.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 8 (Top-Left Panel): Galaxy Image Cutouts Mapped Across UMAP Space
def umap_plotter_images(embeds, images, ranges=[-8, 25, -15, 21], nplt=40, dist_threshold=0.3):
    fig = plt.figure(figsize=(nplt*2, nplt*2))
    spec2 = gridspec.GridSpec(ncols=nplt, nrows=nplt, figure=fig, hspace=0, wspace=0)
    for ii, i in enumerate(np.linspace(ranges[0], ranges[1], nplt)):
        for jj, j in enumerate(np.linspace(ranges[2], ranges[3], nplt)):
            dij = np.argmin(np.abs(embeds[:, 0] - i) + np.abs(embeds[:, 1] - j))
            dist = (np.abs(embeds[dij, 0] - i) + np.abs(embeds[dij, 1] - j))
            if dist < dist_threshold:
                sel_image = Image.open(io.BytesIO(images[dij]))
                fig.add_subplot(spec2[nplt-1-jj, ii]).imshow(sel_image)
                plt.axis('off')
    # plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/umap_vit_embeddings_finetuned_images.pdf', dpi=300, bbox_inches='tight')            
    plt.show()

umap_plotter_images(embedding, gswlc_images_skyserver, nplt=40, dist_threshold=0.3)

In [ ]:
# Figure 8 (Top-Right Panel): Median SFH Trajectories Mapped Across UMAP Space
nsfh = 7

def get_t50_color(t50):
    t50_scaled = min((t50) / 13.25, 1.0)
    return plt.cm.berlin(t50_scaled)

def get_sfh(gal_id):
    sfh = gswlc_sfh_mod[gal_id][0]
    timeax = gswlc_sfh_mod[gal_id][1]
    ssfr = gswlc_data['logSSFR_50'][gal_id] - 9
    return sfh, timeax, ssfr, gswlc_data['t_50'][gal_id]

def umap_plotter_sfh(embeds, ranges=[-8, 25, -15, 21], nplt=40, dist_threshold=0.3):
    fig = plt.figure(figsize=(nplt*2, nplt*2))
    spec2 = gridspec.GridSpec(ncols=nplt, nrows=nplt, figure=fig, hspace=0, wspace=0)
    for ii, i in tqdm(enumerate(np.linspace(ranges[0], ranges[1], nplt))):
        for jj, j in enumerate(np.linspace(ranges[2], ranges[3], nplt)):
            dijs = np.argsort(np.abs(embeds[:, 0] - i) + np.abs(embeds[:, 1] - j))[0:nsfh]
            dist = (np.abs(embeds[dijs[-1], 0] - i) + np.abs(embeds[dijs[-1], 1] - j))
            if dist < dist_threshold:
                sfhs, times, ssfrs, t50s = [], [], [], []
                for npi in range(nsfh):
                    a, b, c, d = get_sfh(dijs[npi])
                    sfhs.append(a / np.sum(a)), times.append(b), ssfrs.append(c), t50s.append(d)
                    
                splined_time = np.linspace(0, np.amax(times), 1000)
                splined_sfhs = [np.interp(splined_time, times[k], np.flip(sfhs[k], 0) / np.amax(sfhs[k])) for k in range(len(sfhs))]
                fig.add_subplot(spec2[nplt-1-jj, ii]).plot(splined_time, np.nanmedian(splined_sfhs, 0), lw=5, color=get_t50_color(np.nanmedian(t50s)))
                plt.fill_between(splined_time, np.nanpercentile(splined_sfhs, 16, 0), np.nanpercentile(splined_sfhs, 84, 0), color=get_t50_color(np.nanmedian(t50s)), alpha=0.1)
                plt.ylim(0, 1)
                plt.axis('off')
    # plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/umap_vit_embeddings_finetuned_sfh.pdf', dpi=300, bbox_inches='tight')
    plt.show()

umap_plotter_sfh(embedding, nplt=40)

# Figure 9: Dusty Starburst Galaxies ($A_v > 3.5$) Degeneracy Breaking

In [ ]:
# Figure 9 (Top-Left & Bottom-Left Panels): M*-SFR Distribution & Error Difference Histogram
labels = ['log M$_*$', 'log sSFR', 'redshift', 't$_{50}$', 'Av', '12+log(O/H)']

better_img = np.where(
    (truth_eval_5b[:, 4] > 3.5) &
    (allsederr_5_bands_full_new[:, 1] - allimerr_full_new[:, 1] > 0) &
    (allsederr_5_bands_full_new[:, 3] - allimerr_full_new[:, 3] > 0) &
    (allsederr_5_bands_full_new[:, 4] - allimerr_full_new[:, 4] > 0) &
    (allsederr_5_bands_full_new[:, 5] - allimerr_full_new[:, 5] > 0)
)[0]

better_sed = np.where(
    (truth_eval_5b[:, 4] > 3.5) &
    (allsederr_5_bands_full_new[:, 1] - allimerr_full_new[:, 1] < 0) &
    (allsederr_5_bands_full_new[:, 3] - allimerr_full_new[:, 3] < 0) &
    (allsederr_5_bands_full_new[:, 4] - allimerr_full_new[:, 4] < 0) &
    (allsederr_5_bands_full_new[:, 5] - allimerr_full_new[:, 5] < 0)
)[0]

print(f"Better Image: {len(better_img)} | Total Av > 3.5: {len(np.where(truth_eval_5b[:, 4] > 3.5)[0])} | Better SED: {len(better_sed)}")

plt.figure(figsize=(8, 6))
plt.scatter(x_m_5b[better_img], y_s_5b[better_img], color='dodgerblue', alpha=1, label='Better Accuracy from Images')
plt.scatter(x_m_5b, y_s_5b, color='red', alpha=0.005)
plt.plot(x_m_5b, 0.83 * x_m_5b - 8.33, color='k', linestyle='--', label='He+26')
plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18)
plt.ylabel('log SFR  [$M_{\odot}$/yr]', fontsize=18)
plt.legend(loc='upper left', fontsize=10)
plt.show()

plt.figure(figsize=(8, 6))
for i in [1, 3, 4, 5]:
    plt.hist(allsederr_5_bands_full_new[:, i][better_img] - allimerr_full_new[:, i][better_img], label=labels[i], alpha=0.5)
plt.legend(loc='upper right', fontsize=14)
plt.xlabel('Photometry - Image Error', fontsize=18)
plt.show()

morph_eval_5b = gswlc_data_merged['bulge_n_morph'] if n_eval_5b == len(gswlc_data_merged) else gswlc_data_merged['bulge_n_morph'].values[train_index:]
print("Median Sersic n (Better Image):", np.nanmedian(np.array(morph_eval_5b)[better_img]))
print("Median Sersic n (Better SED):  ", np.nanmedian(np.array(morph_eval_5b)[better_sed]))

In [ ]:
# Figure 9 (Right Panel): Corner Plot of Centered Residuals for Dusty Starbursts (log sSFR, t50, Av, 12+log(O/H))
samples_im_for_dusty = all_samples_image[:, :, 0:6] if len(all_samples_image) == n_eval_5b else all_samples_image[train_index:, :, 0:6]
covs_im_for_dusty = all_covs_image[:, 0:5, 0:5] if len(all_covs_image) == n_eval_5b else all_covs_image[train_index:, 0:5, 0:5]

subset_bands = get_pooled_residuals(all_samples_5_band[better_img, :, 0:6])
subset_combined = get_pooled_residuals(samples_im_for_dusty[better_img])

if FAST_MODE:
    subset_bands = subset_bands[::STRIDE]
    subset_combined = subset_combined[::STRIDE]

median_cov_bands = np.nanmedian(all_covs_5_band[:, 0:5, 0:5], axis=0)
median_cov_combined = np.nanmedian(covs_im_for_dusty, axis=0)

indices_to_plot = [1, 3, 4, 5]
n_params_subset = len(indices_to_plot)

subset_bands = subset_bands[:, indices_to_plot]
subset_combined = subset_combined[:, indices_to_plot]

subset_labels = [r'$\Delta$ log sSFR', r'$\Delta$ t$_{50}$', r'$\Delta$ Av', r'$\Delta$ 12+log(O/H)'] 
color_bands = 'dodgerblue'
color_combined = 'red'
plot_range = [(-2, 2), (-6, 6), (-2.5, 2.5), (-1.5, 1.5)] 

kwargs_bands = {
    "labels": subset_labels,
    "label_kwargs": {"fontsize": 24},
    "hist_kwargs": {"density": True, "linewidth": 2},
    "plot_datapoints": False,
    "plot_density": True,
    "fill_contours": True,
    "alpha": 0.3,                 
    "levels": (0.68, 0.95),
    "range": plot_range,          
    "bins": 30,                   
}

kwargs_combined = {
    "labels": subset_labels,
    "hist_kwargs": {"density": True, "linewidth": 2.5},
    "plot_datapoints": False,
    "plot_density": False,        
    "fill_contours": False,       
    "contour_kwargs": {"linewidths": 2.5}, 
    "levels": (0.68, 0.95),
    "range": plot_range,          
    "bins": 30,
}

fig = plt.figure(figsize=(22, 12))
fig = corner.corner(subset_bands, fig=fig, color=color_bands, **kwargs_bands)
corner.corner(subset_combined, fig=fig, color=color_combined, **kwargs_combined)

axes = np.array(fig.axes).reshape((n_params_subset, n_params_subset))
bbox_props = dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.8)

for yi in range(n_params_subset):
    for xi in range(yi):
        ax = axes[yi, xi]
        ax.axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.axhline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.plot(0, 0, marker="s", color="black", markersize=3, zorder=5)
        
        orig_y = indices_to_plot[yi]
        orig_x = indices_to_plot[xi]
        
        rho_bands = median_cov_bands[orig_y - 1, orig_x]
        rho_comb = median_cov_combined[orig_y - 1, orig_x]
        
        txt_bands = f"$\\rho_{{phot}} = {rho_bands:.2f}$"
        txt_comb = f"$\\rho_{{comb}} = {rho_comb:.2f}$"
        
        ax.text(0.05, 0.96, txt_bands, transform=ax.transAxes, 
                fontsize=16, color=color_bands, ha='left', va='top', weight='bold', bbox=bbox_props)
        ax.text(0.05, 0.86, txt_comb, transform=ax.transAxes, 
                fontsize=16, color=color_combined, ha='left', va='top', weight='bold', bbox=bbox_props)
    
    axes[yi, yi].axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)

legend_handles = [
    mlines.Line2D([], [], color=color_bands, lw=4, label='5 Band Photometry'),
    mlines.Line2D([], [], color=color_combined, lw=3, label='Image')
]
fig.axes[n_params_subset - 1].legend(handles=legend_handles, loc='center left', fontsize=24, frameon=False)
plt.show()

In [ ]:
# Figure 9 (Bottom Panel): 4x22 Grid of 88 Random Dusty Starburst Galaxy Cutouts
np.random.seed(256)
images_for_dusty = gswlc_images_skyserver if n_eval_5b == len(gswlc_images_skyserver) else gswlc_images_skyserver[train_index:]
selected_sample = np.random.choice(better_img, 88, replace=False)
plt.subplots(4, 22, figsize=(22, 4))
for i in range(len(selected_sample)):
    j = selected_sample[i]
    plt.subplot(4, 22, i + 1)
    plt.imshow(Image.open(io.BytesIO(images_for_dusty[j])))
    plt.axis('off')
plt.tight_layout()
plt.show()

# Appendix B — Figure 10: Random Test Set Galaxies Inferred Posteriors

In [ ]:
ranges = [(9, 12), (-4.5, 3), (0, 0.16), (0, 13.7), (0, 3), (6.5, 9.5)]
labels = ['log M$_*$', 'log sSFR', 'redshift', 't$_{50}$', 'Av', '12+log(O/H)']

conds = {
    'Old Elliptical': (theta[:, 0] > 11.0) & (theta[:, 1] < -2.5) & (theta[:, 4] < 0.5),
    'Young Starburst': (theta[:, 1] > 0.0) & (theta[:, 3] < 3.0),
    'Blue Spiral': (theta[:, 0] > 10.0) & (theta[:, 0] < 10.8) & (theta[:, 1] > -1.0) & (theta[:, 4] < 1.0),
    'Red Spiral': (theta[:, 0] > 10.5) & (theta[:, 1] > -2.5) & (theta[:, 1] < -1.5) & (theta[:, 4] > 1.0),
    'Dusty Edge-on': (theta[:, 4] > 2.0),
    'Green Valley': (theta[:, 1] > -2.0) & (theta[:, 1] < -1.0) & (theta[:, 0] > 10.0)
}

colors = ['firebrick', 'blue', 'cyan', 'salmon', 'saddlebrown', 'olive']
names = list(conds.keys())
test_mask = np.arange(train_index, len(gswlc_data))
gids = []

np.random.seed(177)
for name in names:
    valid_idx = np.where(conds[name])[0]
    valid_test_idx = np.intersect1d(valid_idx, test_mask)
    if len(valid_test_idx) > 0:
        gids.append(np.random.choice(valid_test_idx))
    else:
        print(f"Warning: No match found for {name}. Falling back to a random test galaxy.")
        gids.append(np.random.choice(test_mask))

fig_main = plt.figure(figsize=(24, 16))
subfigs = fig_main.subfigures(2, 3, wspace=0.05, hspace=0.05).flatten()

for i, (gid, name, color) in enumerate(zip(gids, names, colors)):
    xobs_im = gswlc_finetuned_embeds[gid, :]
    x_truth = theta[gid, :]
    
    # Sample posterior and slice [:, 0:6] for the 6 physical parameters
    posterior_samples_im = posterior_ensemble_finetuned_vit.sample(
        (1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False
    ).cpu().numpy()[:, 0:6]
    
    corner.corner(
        posterior_samples_im, 
        fig=subfigs[i], 
        truths=x_truth, 
        truth_color='k', 
        color=color, 
        range=ranges, 
        labels=labels, 
        plot_datapoints=False, 
        fill_contours=True,
        quantiles=(0.16, 0.5, 0.84),
        levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)]
    )
    
    for ax in subfigs[i].get_axes():
        if ax.xaxis.label:
            ax.xaxis.label.set_size(14)
        if ax.yaxis.label:
            ax.yaxis.label.set_size(14)
        for tick in ax.get_xticklabels():
            tick.set_fontsize(10)
            tick.set_rotation(45)
        for tick in ax.get_yticklabels():
            tick.set_fontsize(10)

    ax_img = subfigs[i].add_axes([0.65, 0.65, 0.3, 0.3])
    img = Image.open(io.BytesIO(gswlc_images_skyserver[gid]))
    ax_img.imshow(img)
    ax_img.axis('off')

# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/individual_corner_plots.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Appendix C — Figure 11: 11-Band Photometry vs. Image vs. Combined Prediction Precision

In [ ]:
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

n_plot_11b = len(primary_individual_scatters_11_bands["11 Band Photometry"])
x_m_11b = mvals_full if n_plot_11b == len(mvals_full) else mvals
y_s_11b = sfrvals_full if n_plot_11b == len(sfrvals_full) else sfrvals
vit_scatters_11b = primary_individual_scatters["Finetuned ViT"] if len(primary_individual_scatters["Finetuned ViT"]) == n_plot_11b else primary_individual_scatters["Finetuned ViT"][train_index:]

plt.subplots(figsize=(15, 21), nrows=6, ncols=3)
plt.suptitle('11 Band Photometry vs Image vs Combined Prediction Accuracy', fontsize=18, y=1)
for qty in range(6):
    c_lower_bound = np.nanpercentile(primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], 2)
    c_upper_bound = np.nanpercentile(primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], 98)
    
    plt.subplot(6, 3, qty*3+1)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=primary_individual_scatters_11_bands["11 Band Photometry"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18) 
    plt.ylabel('log SFR  [$M_{\odot}$/yr]', fontsize=18)
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Photometry}$', fontsize=18)
    
    plt.subplot(6, 3, qty*3+2)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=vit_scatters_11b[:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18)  
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Image}$', fontsize=18)
    plt.tight_layout()

    plt.subplot(6, 3, qty*3+3)
    plt.hexbin(x_m_11b, y_s_11b, gridsize=20, mincnt=2, C=primary_individual_scatters_11_bands["11 Band Combined"][:, qty], vmin=c_lower_bound, vmax=c_upper_bound, cmap='viridis')
    if qty == 5:
        plt.xlabel('log $M_*$ [$M_{\odot}$]', fontsize=18) 
    clbr = plt.colorbar()
    clbr.set_label('$\sigma$' + labels[qty] + '$_{,Photometry+Image}$', fontsize=18)

# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/11_bands_SED_vs_image_vs_combined.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Appendix D — Figure 12: 11-Band Photometry, Image, and Combined Inferred Posteriors

In [ ]:
ranges = [(9, 12), (-4.5, 3), (0, 0.16), (0, 13.7), (0, 3), (6.5, 9.5)]
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

np.random.seed(256)
random_gal = np.random.randint(train_index, len(gswlc_data), size=(1))
gid = int(random_gal[0])

xobs_sed = eleven_bands_SED[gid, :]
xobs_im = gswlc_finetuned_embeds[gid, :]
x_truth = theta[gid, :]

# Sample and slice [:, 0:6] for the 6 physical parameters
# posterior_samples_sed = posterior_ensemble_sed_11_bands.sample((1000,), torch.Tensor(xobs_sed).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
# posterior_samples_im = posterior_ensemble_finetuned_vit.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
# posterior_samples_combined = posterior_ensemble_combined_11_bands.sample((1000,), torch.Tensor(np.hstack((xobs_im, xobs_sed))).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]

posterior_samples_sed = posterior_ensemble_sed_11_bands.sample((1000,), torch.Tensor(xobs_sed).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
posterior_samples_im = posterior_ensemble_finetuned_vit.sample((1000,), torch.Tensor(xobs_im).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]
posterior_samples_combined = posterior_ensemble_combined_11_bands.sample((1000,), torch.Tensor(np.hstack((xobs_im, xobs_sed))).to(device), show_progress_bars=False).cpu().numpy()[:, 0:6]

plt.imshow(Image.open(io.BytesIO(gswlc_images_skyserver[gid])))
plt.axis('off')
plt.figure()
plt.scatter(np.arange(11), xobs_sed)
plt.xticks(np.arange(0, 11), ['FUV', 'NUV', 'u', 'g', 'r', 'i', 'z', 'W1', 'W2', 'W3', 'W4'], fontsize=16)
plt.xlabel('Band', fontsize=18)
plt.ylabel('Magnitude', fontsize=18)

fig = corner.corner(posterior_samples_sed, truths=x_truth, color='dodgerblue', range=ranges, labels=labels, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])
fig = corner.corner(posterior_samples_im, truths=x_truth, truth_color='k', color='firebrick', range=ranges, labels=labels, fig=fig, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])
fig = corner.corner(posterior_samples_combined, truths=x_truth, truth_color='k', color='green', range=ranges, labels=labels, fig=fig, plot_datapoints=False, fill_contours=True, quantiles=(0.16, 0.5, 0.84), levels=[1 - np.exp(-(1/1)**2/2), 1 - np.exp(-(2/1)**2/2)])

sed_legend_patch = mpatches.Patch(color='dodgerblue', label='Photometry')
image_legend_patch = mpatches.Patch(color='firebrick', label='Image')
combined_legend_patch = mpatches.Patch(color='green', label='Combined')
fig.legend(handles=[sed_legend_patch, image_legend_patch, combined_legend_patch],
           loc='upper right', fontsize='18', title='Posterior Source', frameon=True, title_fontsize='18')
fig.set_size_inches(12, 12)
for label in fig.get_axes():
    label.xaxis.label.set_size(18)
    label.yaxis.label.set_size(18)
    for tick in label.get_xticklabels():
        tick.set_fontsize(14)
    for tick in label.get_yticklabels():
        tick.set_fontsize(14)
plt.show()

# Appendix E — Figure 13: Population-Level Full Degeneracies (Centered Residuals)

In [ ]:
FAST_MODE = False
STRIDE = 50       

# 1. Compute pooled residuals on the 6 physical parameters ([:, :, 0:6])
res_bands = get_pooled_residuals(all_samples_5_band[:, :, 0:6])
res_combined = get_pooled_residuals(all_samples_combined_5_band[:, :, 0:6])
res_images = get_pooled_residuals(all_samples_image[:, :, 0:6])

if FAST_MODE:
    print(f"Running in FAST_MODE. Plotting 1/{STRIDE} of the samples...")
    res_bands = res_bands[::STRIDE]
    res_combined = res_combined[::STRIDE]
    res_images = res_images[::STRIDE]

# 2. Extract Pre-calculated Median Covariances ([:, 0:5, 0:5])
median_cov_bands = np.nanmedian(all_covs_5_band[:, 0:5, 0:5], axis=0)
median_cov_combined = np.nanmedian(all_covs_combined_5_band[:, 0:5, 0:5], axis=0)
median_cov_images = np.nanmedian(all_covs_image[:, 0:5, 0:5], axis=0)

# 3. Setup plotting parameters
n_params = res_bands.shape[-1]
labels = ['log M$_*$', 'log sSFR', 'redshift', 't50', 'Av', '12+log(O/H)']

color_bands = 'dodgerblue'
color_combined = 'green'
color_images = 'red'

kwargs_bands = {
    "labels": labels,
    "label_kwargs": {"fontsize": 32},
    "hist_kwargs": {"density": True, "linewidth": 2, "histtype": "stepfilled", "alpha": 0.3},
    "plot_datapoints": False,
    "plot_density": True,
    "fill_contours": True,
    "alpha": 0.3,                 
    "levels": (0.68, 0.95),         
    "bins": 30,                   
}

kwargs_combined = {
    "labels": labels,
    "hist_kwargs": {"density": True, "linewidth": 2.5, "histtype": "step", "linestyle": "-"},
    "plot_datapoints": False,
    "plot_density": False,        
    "fill_contours": False,       
    "contour_kwargs": {"linewidths": 2.5, "linestyles": "-"}, 
    "levels": (0.68, 0.95),           
    "bins": 30,
}

kwargs_images = {
    "labels": labels,
    "hist_kwargs": {"density": True, "linewidth": 2.5, "histtype": "step", "linestyle": "--"},
    "plot_datapoints": False,
    "plot_density": False,        
    "fill_contours": False,       
    "contour_kwargs": {"linewidths": 2.5, "linestyles": "--"}, 
    "levels": (0.68, 0.95),           
    "bins": 30,
}

# 4. Generate the corner plot
fig = plt.figure(figsize=(20, 20))
fig = corner.corner(res_bands, fig=fig, color=color_bands, **kwargs_bands)
corner.corner(res_combined, fig=fig, color=color_combined, **kwargs_combined)
corner.corner(res_images, fig=fig, color=color_images, **kwargs_images)

fig.subplots_adjust(wspace=0.2, hspace=0.2)

axes = np.array(fig.axes).reshape((n_params, n_params))
bbox_props = dict(boxstyle="round,pad=0.1", facecolor="white", edgecolor="none", alpha=0.8)

for yi in range(n_params):
    for xi in range(yi):
        ax = axes[yi, xi]
        ax.axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.axhline(0, color="black", linestyle="--", alpha=0.5, zorder=0)
        ax.plot(0, 0, marker="s", color="black", markersize=3, zorder=5)
        
        rho_bands = median_cov_bands[yi - 1, xi]
        rho_comb = median_cov_combined[yi - 1, xi]
        rho_images = median_cov_images[yi - 1, xi]
        
        txt_bands = f"$\\rho_{{phot}} = {rho_bands:.2f}$"
        txt_images = f"$\\rho_{{images}} = {rho_images:.2f}$"
        txt_comb = f"$\\rho_{{comb}} = {rho_comb:.2f}$"

        ax.text(0.05, 0.85, txt_bands, transform=ax.transAxes, 
                fontsize=18, color=color_bands, ha='left', va='bottom', weight='bold', bbox=bbox_props)
        ax.text(0.05, 0.75, txt_images, transform=ax.transAxes, 
                fontsize=18, color=color_images, ha='left', va='bottom', weight='bold', bbox=bbox_props)
        ax.text(0.05, 0.65, txt_comb, transform=ax.transAxes, 
                fontsize=18, color=color_combined, ha='left', va='bottom', weight='bold', bbox=bbox_props)
    
    axes[yi, yi].axvline(0, color="black", linestyle="--", alpha=0.5, zorder=0)

legend_handles = [
    mlines.Line2D([], [], color=color_bands, lw=4, alpha=0.6, label='5 Band Photometry'),
    mlines.Line2D([], [], color=color_images, lw=3, linestyle='--', label='Image'),
    mlines.Line2D([], [], color=color_combined, lw=3, linestyle='-', label='Combined')
]
fig.axes[n_params - 1].legend(handles=legend_handles, loc='center left', fontsize=34, frameon=False)
# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/population_full_degeneracies.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Appendix F — Figure 14: Individual Galaxies SFHs with Confidence Intervals

In [ ]:
gal_mass = mvals_full[slice_im]
gal_sfr  = sfrvals_full[slice_im]
gal_ssfr = gal_sfr - gal_mass
images_sub = gswlc_images_skyserver[slice_im]

def safe_log(arr):
    return np.log10(np.clip(arr, 1e-4, None))

np.random.seed(144)

fig, axes = plt.subplots(4, 3, figsize=(18, 16), sharex=True, sharey='row')
common_lookback_grid = np.linspace(0, 10.0, 200)

print("Generating 10 Gyr Reconstruction Grid (Single Galaxy CIs + Images)...")

for row_idx, ssfr_idx in enumerate([3, 2, 1, 0]): 
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]
        
        m_min, m_max = mass_bins[col_idx], mass_bins[col_idx+1]
        s_min, s_max = ssfr_bins[ssfr_idx], ssfr_bins[ssfr_idx+1]
        
        bin_mask = (gal_mass >= m_min) & (gal_mass < m_max) & \
                   (gal_ssfr >= s_min) & (gal_ssfr < s_max)
        
        indices = np.where(bin_mask)[0]
        n_count = len(indices)
        bin_title = f"{mass_labels[col_idx]}\n{ssfr_labels[ssfr_idx]}\n(Total Bin N={n_count})"

        if n_count == 0:
            ax.text(0.5, 0.5, "No Galaxies", ha='center', transform=ax.transAxes)
            ax.set_title(bin_title, fontsize=10)
            continue

        rand_gal_idx = np.random.choice(indices)

        # Process True SFH
        sfr_true = sfh_mod_sub[rand_gal_idx, 0, :]  
        t_cosmic = sfh_mod_sub[rand_gal_idx, 1, :]
        t_obs = np.max(t_cosmic) 
        t_lookback_gal = t_obs - t_cosmic
        
        sort_order = np.argsort(t_lookback_gal)
        true_sfr_interp = np.interp(common_lookback_grid, 
                                    t_lookback_gal[sort_order], 
                                    sfr_true[sort_order])

        # Process Predicted Posterior Samples (directly in log SFR space from 34D theta)
        pred_med, pred_p16, pred_p84 = [], [], []
        for t in reconstruction_times:
            feature_idx = time_to_idx_34d[t]
            full_sfr_samples = all_samples_image[rand_gal_idx, :, feature_idx]
            pred_med.append(np.nanpercentile(full_sfr_samples, 50))
            pred_p16.append(np.nanpercentile(full_sfr_samples, 16))
            pred_p84.append(np.nanpercentile(full_sfr_samples, 84))

        pred_med = np.array(pred_med)
        pred_p16 = np.array(pred_p16)
        pred_p84 = np.array(pred_p84)

        ax.plot(common_lookback_grid, safe_log(true_sfr_interp), color='black', lw=2, label='True SFH')
        ax.fill_between(reconstruction_times, pred_p16, pred_p84, 
                        color='red', alpha=0.3, label='Predicted 16-84% CI')
        ax.plot(reconstruction_times, pred_med, color='red', 
                linestyle='--', marker='o', markersize=3, label='Predicted Median')

        # Image Inset
        axins = ax.inset_axes([0.65, 0.65, 0.3, 0.3])
        img_bytes = images_sub[rand_gal_idx]
        img_data = np.array(Image.open(io.BytesIO(img_bytes)))
        axins.imshow(img_data, origin='lower') 
        axins.axis('off')
        for spine in axins.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(0.5)

        ax.set_title(bin_title, fontsize=14)
        ax.set_xlim(-0.2, 10.2)
        
        if row_idx == 0: ax.set_ylim(-4.5, 2)
        if row_idx == 1: ax.set_ylim(-2, 2)
        if row_idx == 2: ax.set_ylim(-2, 2)
        if row_idx == 3: ax.set_ylim(-4, 2)
        
        ax.grid(alpha=0.3)
        ax.set_xlabel("Lookback Time [Gyr]", fontsize=18)
        ax.set_ylabel("log SFR [$M_\odot$/yr]", fontsize=18)
        ax.tick_params(labelbottom=True, labelleft=True)
        
        if row_idx == 0 and col_idx == 0:
            ax.legend(loc='lower left', fontsize=14)

plt.tight_layout()
# plt.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/single_galaxy_sfh.pdf', dpi=300)
plt.show()

# Appendix G — Figure 15: Deepest Accurate SFH Prediction (Robust Morphological Memory)

In [ ]:
num_gals = len(x_mass_im)
num_times = len(timescale_map)

print("Calculating Robust Memory Depth from 'individual_accurate_cami' dictionary...")

raw_accuracy_matrix = np.zeros((num_gals, num_times), dtype=bool)
for i, (t_val, key) in enumerate(timescale_map):
    if key in individual_accurate_cami:
        valid_indices = individual_accurate_cami[key]
        raw_accuracy_matrix[valid_indices, i] = True

robust_accuracy_matrix = np.zeros_like(raw_accuracy_matrix)
for i in range(1, num_times - 1):
    robust_accuracy_matrix[:, i] = (
        raw_accuracy_matrix[:, i - 1] & 
        raw_accuracy_matrix[:, i] & 
        raw_accuracy_matrix[:, i + 1]
    )

robust_accuracy_matrix[:, 0] = raw_accuracy_matrix[:, 0] & raw_accuracy_matrix[:, 1]
robust_accuracy_matrix[:, -1] = raw_accuracy_matrix[:, -1] & raw_accuracy_matrix[:, -2]

robust_memory_depth = np.zeros(num_gals)
for i, (t_val, key) in enumerate(timescale_map):
    robust_indices = np.where(robust_accuracy_matrix[:, i])[0]
    robust_memory_depth[robust_indices] = t_val

fig, ax = plt.subplots(figsize=(12, 9))
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
ax.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')

hb = ax.hexbin(x_mass_im, y_sfr_im, C=robust_memory_depth, 
               gridsize=30, cmap='Spectral_r', reduce_C_function=np.mean,
               mincnt=5, vmin=0.0, vmax=10.0, edgecolors='none')

cb = fig.colorbar(hb, ax=ax)
cb.set_label('Lookback Time [Gyr]', fontsize=18) 
cb.set_ticks([0, 2, 4, 6, 8, 10])

ax.set_xlabel(r'$\log M_* \ [M_\odot]$', fontsize=18)
ax.set_ylabel(r'$\log \mathrm{SFR} \ [M_\odot/\mathrm{yr}]$', fontsize=18)
ax.set_title('Deepest Accurate SFH Prediction', fontsize=18)
ax.legend(loc='lower right', frameon=False, labelcolor='black') 
ax.set_ylim(-5, 3.5)
ax.set_xlim(7.8, 12)
# fig.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/robust_memory_depth_CI_colour_flipped.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Appendix H — Figure 16: Continuous SFH Memory from 11-Band and 5-Band Photometry Models

In [ ]:
# Compute SFH Memory Dictionaries for 11-Band and 5-Band Photometry Models
truth_11b_34d = theta_full_sfh_sfr if len(all_samples_11_band) == len(theta_full_sfh_sfr) else theta_full_sfh_sfr[train_index:]
individual_accurate_11_bands, cumulative_accurate_11_bands, failed_at_step_11_bands = compute_sfh_memory_dicts(all_samples_11_band, truth_11b_34d)

truth_5b_34d = theta_full_sfh_sfr if len(all_samples_5_band) == len(theta_full_sfh_sfr) else theta_full_sfh_sfr[train_index:]
individual_accurate_5_bands, cumulative_accurate_5_bands, failed_at_step_5_bands = compute_sfh_memory_dicts(all_samples_5_band, truth_5b_34d)

In [ ]:
# Figure 16 (Top Panel): Continuous SFH Memory (11-Band Photometry)
memory_depth_11_bands = np.zeros(len(x_m_11b))
for t_val, key in timescale_map:
    if key in cumulative_accurate_11_bands:
        memory_depth_11_bands[cumulative_accurate_11_bands[key]] = t_val

fig, ax = plt.subplots(figsize=(10, 8))
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
ax.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')

hb = ax.hexbin(x_m_11b, y_s_11b, C=memory_depth_11_bands, 
               gridsize=30, cmap='Spectral_r', reduce_C_function=np.mean,
               mincnt=5, vmin=0.0, vmax=8, edgecolors='none')

cb = fig.colorbar(hb, ax=ax, extend='both')
cb.set_label('Lookback Time [Gyr]', fontsize=18) 

ax.set_xlabel(r'$\log M_* \ [M_\odot]$', fontsize=18)
ax.set_ylabel(r'$\log \mathrm{SFR} \ [M_\odot/\mathrm{yr}]$', fontsize=18)
ax.set_title('Continuously Accurate SFH Prediction (11 Band Photometry)', fontsize=18)
ax.legend(loc='lower right')
ax.set_ylim(-5, 3.5)
ax.set_xlim(7.8, 12)
# fig.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/morphological_memory_depth_11_bands.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 16 (Bottom Panel): Continuous SFH Memory (5-Band Photometry)
memory_depth_5_bands = np.zeros(len(x_m_5b))
for t_val, key in timescale_map:
    if key in cumulative_accurate_5_bands:
        memory_depth_5_bands[cumulative_accurate_5_bands[key]] = t_val

fig, ax = plt.subplots(figsize=(10, 8))
ms_x = np.linspace(7.5, 12, 100)
ms_y = 0.83 * ms_x - 8.33
ax.plot(ms_x, ms_y, 'k--', lw=2, label='Main Sequence (He+26)')

hb = ax.hexbin(x_m_5b, y_s_5b, C=memory_depth_5_bands, 
               gridsize=30, cmap='Spectral_r', reduce_C_function=np.mean,
               mincnt=5, vmin=0.0, vmax=8, edgecolors='none')

cb = fig.colorbar(hb, ax=ax, extend='both')
cb.set_label('Lookback Time [Gyr]', fontsize=18) 

ax.set_xlabel(r'$\log M_* \ [M_\odot]$', fontsize=18)
ax.set_ylabel(r'$\log \mathrm{SFR} \ [M_\odot/\mathrm{yr}]$', fontsize=18)
ax.set_title('Continuously Accurate SFH Prediction (5 Band Photometry)', fontsize=18)
ax.legend(loc='lower right')
ax.set_ylim(-5, 3.5)
ax.set_xlim(7.8, 12)
# fig.savefig('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/Plots/morphological_memory_depth_5_bands.pdf', dpi=300, bbox_inches='tight')
plt.show()